# Cross-Sectional Momentum on S&P 500

**Research question:** Do past winners continue to outperform past losers in the S&P 500?

**Hypothesis:** Stocks with the highest returns over the past 12 months will outperform stocks with the lowest returns over the following month.

In [48]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')

## 1. Data Collection
Download daily adjusted close prices for all S&P 500 constituents.

In [3]:
response = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
data_frame = pd.read_csv(pd.io.common.StringIO(response.text))

list_sp500 = data_frame["Symbol"].tolist()
print(list_sp500[:10])

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


## 2. Download Historical Prices
We download daily adjusted close prices for all S&P 500 constituents since 2010. Adjusted close accounts for dividends and stock splits, ensuring consistent return calculations.

In [4]:
# Fix Yahoo Finance ticker format
list_sp500 = [t.replace('.', '-') for t in list_sp500]

data_candles = yf.download(list_sp500, "2010-01-01", auto_adjust=True)

print(data_candles.shape)
print(data_candles.head())

[*********************100%***********************]  503 of 503 completed


(4236, 2515)
Price           Close                                                      \
Ticker              A      AAPL ABBV ABNB        ABT      ACGL        ACN   
Date                                                                        
2010-01-04  19.743513  6.400958  NaN  NaN  17.968191  7.601905  30.610769   
2010-01-05  19.529047  6.412026  NaN  NaN  17.823017  7.576549  30.799955   
2010-01-06  19.459663  6.310036  NaN  NaN  17.921997  7.543795  31.127359   
2010-01-07  19.434427  6.298371  NaN  NaN  18.070465  7.499420  31.098267   
2010-01-08  19.428122  6.340243  NaN  NaN  18.162846  7.484628  30.974573   

Price                                        ...   Volume                    \
Ticker           ADBE        ADI        ADM  ...       WY     WYNN      XEL   
Date                                         ...                              
2010-01-04  37.090000  21.703716  20.029287  ...  1832400  4741400  2670400   
2010-01-05  37.700001  21.669453  20.137484  ...  1724

In [5]:
price = data_candles["Close"]
print(price.head())

Ticker              A      AAPL  ABBV  ABNB        ABT      ACGL        ACN  \
Date                                                                          
2010-01-04  19.743513  6.400958   NaN   NaN  17.968191  7.601905  30.610769   
2010-01-05  19.529047  6.412026   NaN   NaN  17.823017  7.576549  30.799955   
2010-01-06  19.459663  6.310036   NaN   NaN  17.921997  7.543795  31.127359   
2010-01-07  19.434427  6.298371   NaN   NaN  18.070465  7.499420  31.098267   
2010-01-08  19.428122  6.340243   NaN   NaN  18.162846  7.484628  30.974573   

Ticker           ADBE        ADI        ADM  ...        WY       WYNN  \
Date                                         ...                        
2010-01-04  37.090000  21.703716  20.029287  ...  9.157351  40.493736   
2010-01-05  37.700001  21.669453  20.137484  ...  9.351745  42.956539   
2010-01-06  37.619999  21.628336  20.086561  ...  9.248345  42.393063   
2010-01-07  36.889999  21.457003  19.876530  ...  9.206983  43.298401   
2010-01-

## 3. Compute Monthly Returns
We resample daily prices to monthly frequency and compute log returns. Log returns are additive over time, which simplifies the calculation of cumulative multi-month returns needed for the momentum signal.

In [6]:
monthly_price = price.resample('ME').last()
monthly_log = np.log(monthly_price)
monthly_yield = monthly_log.diff()
print(monthly_yield.head())

Ticker             A      AAPL  ABBV  ABNB       ABT      ACGL       ACN  \
Date                                                                       
2010-01-31       NaN       NaN   NaN   NaN       NaN       NaN       NaN   
2010-02-28  0.115441  0.063347   NaN   NaN  0.024996  0.033538 -0.025199   
2010-03-31  0.089049  0.138431   NaN   NaN -0.029919  0.030223  0.048349   
2010-04-30  0.052949  0.105280   NaN   NaN -0.020889 -0.008826  0.048235   
2010-05-31 -0.113792 -0.016257   NaN   NaN -0.072966 -0.027634 -0.151100   

Ticker          ADBE       ADI       ADM  ...        WY      WYNN       XEL  \
Date                                      ...                                 
2010-01-31       NaN       NaN       NaN  ...       NaN       NaN       NaN   
2010-02-28  0.070231  0.081183 -0.015512  ...  0.012453  0.026944  0.001443   
2010-03-31  0.020566 -0.007693 -0.015792  ...  0.113815  0.176352  0.030028   
2010-04-30 -0.051338  0.037792 -0.033782  ...  0.090730  0.151566  0.025

## 4. Momentum Signal
For each stock, we compute the cumulative log return over the past 12 months. This is our momentum score: stocks with high past returns are "winners", stocks with low past returns are "losers".

In [7]:
annual_yield = monthly_yield.rolling(12).sum()
print(annual_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-07-31  0.194645  0.401541  0.313912  0.134814 -0.154333  0.155411   
2026-08-31  0.208504  0.314775  0.228358  0.339085 -0.161191  0.069015   
2026-09-30  0.300521  0.272078  0.152577  0.279972 -0.281140  0.017806   
2026-10-31  0.185450  0.246367  0.331414  0.261683 -0.127337  0.208884   
2026-11-30  0.161273  0.240451  0.274337  0.429924 -0.083224  0.156035   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-07-31 -0.455720 -0.356567  0.505358  0.411562  ...  0.033970 -0.084215   
2026-08-31 -0.294409 -0.197439  0.378678  0.289418  ... -0.074988 -0.318965   
2026-09-30 -0.275865 -0.385371  0.491524  0.313044  ... -0.227574 -0.469941   
2026-10-31  0.060416 -0.307476  0.589903  0.268296  ... -0.086614 -0.456775   
2026-11

## 5. Portfolio Construction
Each month, we rank all stocks by their 12-month momentum score and sort them into deciles (10 groups). Decile 1 contains the biggest losers, decile 10 the biggest winners.

In [8]:
ranked_yield = annual_yield.rank(axis=1, pct=True)
print(ranked_yield.tail())

Ticker             A      AAPL      ABBV      ABNB       ABT      ACGL  \
Date                                                                     
2026-07-31  0.594378  0.827309  0.732932  0.508032  0.170683  0.546185   
2026-08-31  0.652610  0.777108  0.684739  0.811245  0.154618  0.451807   
2026-09-30  0.803213  0.769076  0.650602  0.785141  0.136546  0.457831   
2026-10-31  0.595190  0.691383  0.805611  0.713427  0.152305  0.631263   
2026-11-30  0.557114  0.685371  0.751503  0.925852  0.164329  0.545090   

Ticker           ACN      ADBE       ADI       ADM  ...        WY      WYNN  \
Date                                                ...                       
2026-07-31  0.042169  0.072289  0.885542  0.837349  ...  0.359438  0.216867   
2026-08-31  0.076305  0.132530  0.843373  0.761044  ...  0.226908  0.056225   
2026-09-30  0.140562  0.080321  0.911647  0.819277  ...  0.174699  0.058233   
2026-10-31  0.386774  0.076152  0.955912  0.719439  ...  0.190381  0.048096   
2026-11

In [9]:
deciles = np.ceil(ranked_yield * 10)
print(deciles.tail())

Ticker        A  AAPL  ABBV  ABNB  ABT  ACGL  ACN  ADBE   ADI  ADM  ...   WY  \
Date                                                                ...        
2026-07-31  6.0   9.0   8.0   6.0  2.0   6.0  1.0   1.0   9.0  9.0  ...  4.0   
2026-08-31  7.0   8.0   7.0   9.0  2.0   5.0  1.0   2.0   9.0  8.0  ...  3.0   
2026-09-30  9.0   8.0   7.0   8.0  2.0   5.0  2.0   1.0  10.0  9.0  ...  2.0   
2026-10-31  6.0   7.0   9.0   8.0  2.0   7.0  4.0   1.0  10.0  8.0  ...  2.0   
2026-11-30  6.0   7.0   8.0  10.0  2.0   6.0  5.0   1.0  10.0  6.0  ...  3.0   

Ticker      WYNN  XEL   XOM  XYL  XYZ  YUM  ZBH  ZBRA  ZTS  
Date                                                        
2026-07-31   3.0  5.0   8.0  2.0  4.0  5.0  4.0   2.0  1.0  
2026-08-31   1.0  5.0   9.0  2.0  4.0  5.0  3.0   6.0  1.0  
2026-09-30   1.0  3.0   9.0  1.0  5.0  4.0  3.0   8.0  1.0  
2026-10-31   1.0  4.0  10.0  1.0  4.0  6.0  3.0   8.0  1.0  
2026-11-30   1.0  3.0  10.0  1.0  8.0  5.0  6.0   8.0  1.0  

[5 rows x 5

## 6. Backtest
Each month, we go long the top decile (winners) and short the bottom decile (losers). We compute the equal-weighted return of each portfolio and track the long-short spread over time.

In [22]:
signal = deciles.shift(1)
long_ret = monthly_yield.where(signal == 10).mean(axis=1)
short_ret = monthly_yield.where(signal == 1).mean(axis=1)
strategy_ret = long_ret - short_ret
print(strategy_ret.tail())

Date
2026-07-31   -0.262365
2026-08-31   -0.068201
2026-09-30    0.173477
2026-10-31   -0.156157
2026-11-30   -0.112096
Freq: ME, dtype: float64


## 7. Performance Analysis
We compute key performance metrics for the long-short momentum strategy: annualized return, Sharpe ratio, and win rate.

In [29]:
monthly_ret = strategy_ret.mean()
standard_deviation = strategy_ret.std()
positive_months = (strategy_ret > 0).sum()
negative_months = (strategy_ret < 0).sum()
print(monthly_ret)
print(standard_deviation)
print(positive_months)
print(negative_months)

0.004841458930640184
0.062130613429843815
102
88


In [41]:
annual_ret = monthly_ret * 12
annual_std = standard_deviation * np.sqrt(12)
sharpe_ratio = annual_ret / annual_std
print(annual_ret)
print(annual_std)
print(sharpe_ratio)

0.05809750716768221
0.21522675833182142
0.2699362645146175


## 8. Robustness Checks
### 8.1 Alpha Decay Test
We split the sample into two sub-periods to test whether the momentum premium has weakened over time.

In [49]:
early_ret = strategy_ret.loc[strategy_ret.index.year <= 2018]
late_ret = strategy_ret.loc[strategy_ret.index.year > 2018]

monthly_early_ret = early_ret.mean()
monthly_late_ret = late_ret.mean()

annual_early_ret = monthly_early_ret * 12
annual_late_ret = monthly_late_ret * 12

standard_deviation_early = early_ret.std()
standard_deviation_late = late_ret.std()

annual_std_early = standard_deviation_early * np.sqrt(12)
annual_std_late = standard_deviation_late * np.sqrt(12)

early_sharpe_ratio = annual_early_ret / annual_std_early
late_sharpe_ratio = annual_late_ret / annual_std_late

print(f"Early period (2011-2018) Sharpe: {early_sharpe_ratio:.3f}")
print(f"Late period (2019-2026) Sharpe: {late_sharpe_ratio:.3f}")

Early period (2011-2018) Sharpe: 0.190
Late period (2019-2026) Sharpe: 0.331
